# 2.2 — Visualizing Evoked Data

This notebook covers the main visualization tools for `Evoked` objects in MNE-Python,
adapted from the [MNE Visualizing Evoked tutorial](https://mne.tools/stable/auto_tutorials/evoked/20_visualize_evoked.html)
and [ERP analysis tutorial](https://mne.tools/stable/auto_tutorials/evoked/30_eeg_erp.html).

We pick up directly from notebook 2.1. Rather than recomputing everything from epochs,
we load the evoked objects that were saved to disk at the end of that notebook.

**What we will cover:**
1. **Butterfly plots** — all Z-channel traces overlaid, GFP line, highlighted time windows, condition overlay
2. **Scalp topographies** — `plot_topomap()` at specific times and averaged over windows
3. **Joint plots** — butterfly + topographies combined; X, Y, Z components of the MMN separately
4. **Image plots** — one channel per row, amplitude as colour, time on x-axis
5. **Topographical subplots** — one waveform per sensor in a spatial layout via `plot_compare_evokeds(axes='topo')`
6. **3D field maps** — field interpolated onto the MEG helmet or head surface via `mne.make_field_map()`
7. **Amplitude and latency measures** — `get_peak()`, mean amplitude over time windows, GFP summary

---
### Running this notebook

```bash
uv run --project $MNE_OPM_DIR jupyter lab
```

Use `%matplotlib inline` for static figures (default below).  
Switch to `%matplotlib qt` for **interactive** figures where you can click a time
point on the butterfly plot and get a topography pop-up automatically.


## 0. Setup

In [ ]:
%matplotlib inline
# %matplotlib qt  # uncomment for interactive plots

import pathlib
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level('WARNING')
print(f'MNE version: {mne.__version__}')

In [ ]:
# ─── USER SETTINGS ───────────────────────────────────────────────────────────
DERIV_DIR    = pathlib.Path('/path/to/bids/derivatives/analysis1__/sub-001/ses-01/meg')
SUBJECTS_DIR = pathlib.Path('/path/to/bids/derivatives/freesurfer')  # FreeSurfer subjects directory

SUBJECT      = 'sub-001'
SESSION      = 'ses-01'
TASK         = 'oddball'

# FreeSurfer subject name — may differ from SUBJECT if recon-all was run with
# a combined subject+session label (e.g. 'sub-001_ses-01' rather than 'sub-001').
FS_SUBJECT   = f'{SUBJECT}_{SESSION}'   # adjust if your subjects dir uses a different name

STANDARD_COND = 'standard_onset'
DEVIANT_COND  = 'deviant_onset'

# Trans file aligns MEG sensor positions to the head/MRI coordinate system.
# Produced during coregistration (e.g. mne coreg).
TRANS_FILE = DERIV_DIR / f'{SUBJECT}_{SESSION}_task-{TASK}_trans.fif'
# ─────────────────────────────────────────────────────────────────────────────

evoked_fif = DERIV_DIR / f'{SUBJECT}_{SESSION}_task-{TASK}_ave.fif'
print(f'Evoked file  : {evoked_fif}')
print(f'File exists  : {evoked_fif.exists()}')
print(f'Trans file   : {TRANS_FILE}')
print(f'Trans exists : {TRANS_FILE.exists()}')
print(f'FS subject   : {FS_SUBJECT}')
print(f'Subjects dir : {SUBJECTS_DIR / FS_SUBJECT}')
print(f'BEM exists   : {(SUBJECTS_DIR / FS_SUBJECT / "bem").exists()}')


## 1. Load Evoked Objects

We load the evoked objects saved at the end of notebook 2.1. Recall that we saved:
- `evoked_std` — standard condition average (~814 trials)
- `evoked_std_matched` - standard condition, preceding-deviant matched (~168 trials)
- `evoked_dev` — deviant condition average (~168 trials)
- `evoked_mmn_matched` — context-matched MMN difference wave (deviant − preceding standard)

This is the starting point for all visualizations in this notebook.

In [ ]:
all_evoked = mne.read_evokeds(evoked_fif, verbose=False)

evoked_std         = mne.read_evokeds(evoked_fif, condition=STANDARD_COND,                                  verbose=False)
evoked_std_matched = mne.read_evokeds(evoked_fif, condition='standard_onset (preceding-deviant matched)',   verbose=False)
evoked_dev         = mne.read_evokeds(evoked_fif, condition=DEVIANT_COND,                                   verbose=False)
evoked_mmn         = mne.read_evokeds(evoked_fif, condition='MMN context-matched (deviant - preceding standard)', verbose=False)

print('Loaded evoked objects:')
for ev in [evoked_std, evoked_std_matched, evoked_dev, evoked_mmn]:
    print(f'  {ev.comment:<55s}  nave={ev.nave}  shape={ev.data.shape}')

# Note: the MMN difference wave will show nave=84, not 168 — this is expected.
# mne.combine_evoked() computes nave for difference waves using the formula
# 1 / sum(w² / nave_i), which for weights=[1, -1] and nave=168 on both sides gives
# 1 / (1/168 + 1/168) = 84. This reflects the effective sample size of a difference
# score in statistics — not a loss of data or a bug.


## 2. Butterfly Plots

A **butterfly plot** overlays every channel as a separate waveform trace. When you
have many channels (as we do with OPM-MEG), the result looks like a butterfly — a
dense band of traces that spreads and contracts as the response rises and falls.

Butterfly plots are useful for:
- Checking the time course of responses across all sensors at once
- Spotting which channels are most active (the outlying traces)
- Identifying bad channels (traces that sit far outside the others at all times)
- Getting a feel for the response latency, polarity distribution, and SNR

We plot only the **Z (radial) component** here. Z is the component most directly
comparable to traditional SQUID-MEG, and produces the clearest field patterns over
auditory cortex. The X and Y (tangential) components have different spatial patterns
and will be shown in Section 4 alongside Z in the joint plots.

The `highlight` parameter shades time windows of interest. It accepts a list of
`(tmin, tmax)` tuples in seconds — here we mark the N100/M100 window (75–125 ms)
and the MMN window (175-225 ms). Note that all highlighted regions share the same
color; for different colors per window use `axvspan()` calls manually after the plot.

### 2a. Basic butterfly plot with highlighted windows

In [ ]:
# Pick Z-component mag channels, excluding any marked as bad.
# 'mag' selects all magnetometer channels; for OPM data this includes X, Y, and Z.
# exclude='bads' drops any channels marked bad in epochs.info['bads'].
# We then further restrict to channels whose names end in ' Z' (the radial component).
# Because z_names is built from this already-filtered list, bad channels are
# implicitly excluded from all evoked_*_z objects below — even though
# pick_channels() itself does not re-check the bads list.
z_names = [ch for ch in evoked_dev.copy().pick('mag', exclude='bads').ch_names
           if ch.endswith(' Z')]

evoked_std_z = evoked_std.copy().pick_channels(z_names)
evoked_std_matched_z = evoked_std_matched.copy().pick_channels(z_names)
evoked_dev_z = evoked_dev.copy().pick_channels(z_names)
evoked_mmn_z = evoked_mmn.copy().pick_channels(z_names)

print(f'Z channels selected: {len(z_names)}')
print(f'Example names: {z_names[:5]}')

In [ ]:
# Butterfly plot — deviant condition, Z channels
# units/scalings convert from Tesla to femtoTesla for display.
# gfp=True adds a thick green line showing the Global Field Power (GFP) at every
# time point. GFP is the population standard deviation across all sensors: it is
# zero when all sensors agree, and large when sensors differ strongly — so GFP
# peaks mark moments of spatially distributed brain activity. We computed this
# quantity by hand in notebook 2.1 Section 5b; here MNE does it automatically.

fig = evoked_dev_z.plot(
    picks='mag',
    units=dict(mag='fT'),
    scalings=dict(mag=1e15),
    titles=dict(mag=f'Deviant — Z channels (butterfly)'),
    gfp=True,
    highlight=[(0.075, 0.125), (0.175, 0.225)],
    show=False,
)
for ax in fig.axes:
    ax.axvline(0, color='black', lw=0.8, ls='--', zorder=3)
    ax.axvspan(-0.200, 0, color='gray', alpha=0.15, zorder=0)
plt.show()

fig = evoked_std_matched_z.plot(
    picks='mag',
    units=dict(mag='fT'),
    scalings=dict(mag=1e15),
    titles=dict(mag=f'Standard (matched) — Z channels (butterfly)'),
    gfp=True,
    highlight=[(0.075, 0.125), (0.175, 0.225)],
    show=False,
)
for ax in fig.axes:
    ax.axvline(0, color='black', lw=0.8, ls='--', zorder=3)
    ax.axvspan(-0.200, 0, color='gray', alpha=0.15, zorder=0)
plt.show()

fig = evoked_std_z.plot(
    picks='mag',
    units=dict(mag='fT'),
    scalings=dict(mag=1e15),
    titles=dict(mag=f'Standard (all) — Z channels (butterfly)'),
    gfp=True,
    highlight=[(0.075, 0.125), (0.175, 0.225)],
    show=False,
)
for ax in fig.axes:
    ax.axvline(0, color='black', lw=0.8, ls='--', zorder=3)
    ax.axvspan(-0.200, 0, color='gray', alpha=0.15, zorder=0)
plt.show()

### 2c. Overlaying conditions

We compare the **context-matched standard** (the standard trial immediately preceding
each deviant, 168 trials) against the **deviant** (168 trials). Using the
context-matched standard rather than the full standard average (814 trials) gives
a fairer comparison — equal trial counts and better-matched adaptation state.

`mne.viz.plot_compare_evokeds()` is designed for overlaying conditions. It plots
the GFP for each condition as a bold line. Confidence bands and individual channel
traces only appear when multiple evoked objects are passed per condition (e.g. one
per subject in a group study) — with a single evoked per condition, only the GFP
line is shown.

Note: `plot_compare_evokeds` expects a dict of `{label: [evoked]}` with each value
as a **list**.

In [ ]:
# plot_compare_evokeds overlays multiple conditions.
# With a single evoked per condition, each line is the GFP (RMS across channels).
# Confidence bands and individual traces only appear with multiple evokeds per condition.


fig = mne.viz.plot_compare_evokeds(
    {'Standard (matched)': [evoked_std_matched_z],
     DEVIANT_COND:         [evoked_dev_z]},
    picks='mag',
    show=False,
)
plt.show()


## 3. Scalp Topographies

A **scalp topography** (or 'topomap') shows the spatial distribution of field
amplitude at a single moment in time. In OPM-MEG, each sensor's value is interpolated
across the scalp surface to produce a smooth map. Regions of strong positive field
(field emerging from the head) appear warm, and regions of strong negative field
(field entering the head) appear cool.

As with the butterfly plots, we use **Z channels only** for topographies.

### 3a. Interactive topographies from a butterfly plot

When running with `%matplotlib qt` (interactive mode), clicking any time point on
a butterfly plot automatically opens a topomap at that moment. This is the fastest
way to explore the spatial structure of a response — scrub through the time axis
and watch the field pattern evolve.

To try it:
1. Uncomment `%matplotlib qt` in the setup cell and restart the kernel
2. Run the butterfly plot cell above
3. Click anywhere on the waveform

The cells below show how to produce the same topographies **programmatically**,
which is what you need for figures and reports.

### 3b. Topomap at specific time points

In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────────
SHARED_COLORSCALE = True   # True: both topomap rows share the same colorscale
                           # False: MNE scales each plot independently

times_to_plot = [0.050, 0.100, 0.150, 0.200, 0.250, 0.300]

# ── Shared colorscale ─────────────────────────────────────────────────────────
# When SHARED_COLORSCALE=True, compute vmax in fT from both evoked objects at
# the selected time points, so the two plots are directly comparable.
# When False, MNE chooses its own scale per plot.
if SHARED_COLORSCALE:
    # Get data in fT (same scaling as the plots below)
    def _peak_fT(evoked, times):
        data_fT = evoked.copy().pick('mag').get_data() * 1e15
        time_idx = [np.argmin(np.abs(evoked.times - t)) for t in times]
        return np.abs(data_fT[:, time_idx]).max()

    shared_vmax = max(
        _peak_fT(evoked_std_matched_z, times_to_plot),
        _peak_fT(evoked_dev_z,         times_to_plot),
    )
    vlim = (-shared_vmax, shared_vmax)
    print(f'Shared colorscale: ±{shared_vmax:.1f} fT')
else:
    vlim = None   # let MNE decide per plot
    print('Independent colorscale per plot (MNE default)')

fig = evoked_std_matched_z.plot_topomap(
    times=times_to_plot,
    ch_type='mag',
    units=dict(mag='fT'),
    scalings=dict(mag=1e15),
    time_unit='ms',
    vlim=vlim,
    colorbar=True,
    show=False,
)
fig.suptitle('Standard matched — Z channels: field topography at selected time points', y=1.02)
plt.show()

fig = evoked_dev_z.plot_topomap(
    times=times_to_plot,
    ch_type='mag',
    units=dict(mag='fT'),
    scalings=dict(mag=1e15),
    time_unit='ms',
    vlim=vlim,
    colorbar=True,
    show=False,
)
fig.suptitle('Deviant — Z channels: field topography at selected time points', y=1.02)
plt.show()

### 3c. Averaging topographies over a time window

Rather than snapping to a single time point, you can average the field over a
window to get a more stable, SNR-optimized spatial map. This is especially useful
for components like the MMN that have a broad time course.

Pass `average=` with the window duration in seconds and MNE will slide a window
of that width across the times you specify, averaging within each window.

In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────────
SHARED_COLORSCALE = True   # True: all three topomap rows share the same colorscale
                           # False: MNE scales each plot independently

times_to_avg = [0.100, 0.200]
average_win  = 0.050   # 50 ms window centered on each time point

# ── Shared colorscale ─────────────────────────────────────────────────────────
# When SHARED_COLORSCALE=True, compute vmax in fT from all three evoked objects
# at the selected time points so the plots are directly comparable.
# Note: the MMN difference wave is typically much smaller than the raw conditions,
# so including it in the shared scale may compress its colormap — set
# SHARED_COLORSCALE=False to let it scale independently if needed.
if SHARED_COLORSCALE:
    def _peak_fT(evoked, times, average):
        data_fT = evoked.copy().pick('mag').get_data() * 1e15
        vals = []
        for t in times:
            t_mask = (evoked.times >= t - average/2) & (evoked.times <= t + average/2)
            vals.append(np.abs(data_fT[:, t_mask]).mean(axis=1).max())
        return max(vals)

    shared_vmax = max(
        _peak_fT(evoked_std_matched_z, times_to_avg, average_win),
        _peak_fT(evoked_dev_z,         times_to_avg, average_win),
        _peak_fT(evoked_mmn_z,         times_to_avg, average_win),
    )
    vlim = (-shared_vmax, shared_vmax)
    print(f'Shared colorscale: ±{shared_vmax:.1f} fT')
else:
    vlim = None
    print('Independent colorscale per plot (MNE default)')

# average= sets the width of the averaging window in seconds.
# Each topomap shows the mean field over a window of this width centered on
# the corresponding time point.
# Here we use two windows that bracket the MMN component:
#   75–125 ms  — early auditory onset
#   175–225 ms — MMN window
# Note: displayed window edges may differ slightly (e.g. 77–122 ms instead of
# 75–125 ms) — MNE snaps boundaries to the nearest sample (2.5 ms at 400 Hz).

fig = evoked_std_matched_z.plot_topomap(
    times=times_to_avg,
    ch_type='mag',
    average=average_win,
    units=dict(mag='fT'),
    scalings=dict(mag=1e15),
    time_unit='ms',
    vlim=vlim,
    colorbar=True,
    show=False,
)
fig.suptitle('Standard matched — Z channels: field averaged over 50 ms windows', y=1.02)
plt.show()

fig = evoked_dev_z.plot_topomap(
    times=times_to_avg,
    ch_type='mag',
    average=average_win,
    units=dict(mag='fT'),
    scalings=dict(mag=1e15),
    time_unit='ms',
    vlim=vlim,
    colorbar=True,
    show=False,
)
fig.suptitle('Deviant — Z channels: field averaged over 50 ms windows', y=1.02)
plt.show()

# ── Same for the context-matched MMN difference wave ──────────────────────────
# The difference wave isolates the mismatch response — the topomap should show
# a cleaner dipolar pattern than the raw deviant average.
fig = evoked_mmn_z.plot_topomap(
    times=times_to_avg,
    ch_type='mag',
    average=average_win,
    units=dict(mag='fT'),
    scalings=dict(mag=1e15),
    time_unit='ms',
    vlim=vlim,
    colorbar=True,
    show=False,
)
fig.suptitle('Context-matched MMN — Z channels: field averaged over 50 ms windows', y=1.02)
plt.show()

## 4. Joint Plots

A **joint plot** combines a butterfly plot with a row of topographies at selected
time points, all in one figure. It is one of the most compact and informative
single-figure summaries you can make for an evoked response.

`evoked.plot_joint()` automatically detects peaks in the GFP and places topographies
there, or you can specify times manually with `times=`.

### 4a. Z component — deviant and MMN

We start with Z channels (radial component) for both the deviant average and the
context-matched MMN difference wave.

In [ ]:
# plot_joint() — butterfly + topographies in one figure.
# times='peaks' (default) finds GFP peaks automatically.
fig = evoked_mmn_z.plot_joint(
    times=[0.100, 0.150, 0.200, 0.300],
    picks='mag',
    title='Context-matched MMN — Z channels',
    show=False,
)
plt.show()


### 4b. X, Y, Z components compared

OPM sensors are **triaxial** — each sensor measures the magnetic field in three
orthogonal directions simultaneously:
- **Z (radial):** field component pointing in/out of the head. This is the component
  conventional SQUID-MEG primarily measures, and gives the classic dipolar pattern.
- **X (tangential):** one tangential component, lying in the plane of the scalp.
  Tangential components are sensitive to different aspects of the source geometry.
- **Y (tangential):** the other tangential component, orthogonal to X.

All three components carry independent spatial information about the underlying
sources. Comparing their topographies at the same time point gives a richer picture
of the field geometry than Z alone.

Note that the colorscale is component-specific — do not compare absolute values
across the three plots, only the spatial patterns.

In [ ]:
# Build per-component evoked objects from the context-matched MMN.
# Channel names follow the pattern: SENSORNAME X, SENSORNAME Y, SENSORNAME Z
base_names = evoked_mmn.copy().pick('mag', exclude='bads').ch_names

x_names = [ch for ch in base_names if ch.endswith(' X')]
y_names = [ch for ch in base_names if ch.endswith(' Y')]
# z_names already defined above

evoked_mmn_x = evoked_mmn.copy().pick_channels(x_names)
evoked_mmn_y = evoked_mmn.copy().pick_channels(y_names)

print(f'X channels: {len(x_names)}')
print(f'Y channels: {len(y_names)}')
print(f'Z channels: {len(z_names)}')


In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────────
SHARED_COLORSCALE = True   # True: all three joint plots share the same topomap colorscale
                           # False: MNE scales each plot independently

JOINT_TIMES = [0.100, 0.150, 0.200, 0.300]

evokeds = [
    (evoked_mmn_x, 'Context-matched MMN — X (tangential)'),
    (evoked_mmn_y, 'Context-matched MMN — Y (tangential)'),
    (evoked_mmn_z, 'Context-matched MMN — Z (radial)'),
]

# ── Shared colorscale ─────────────────────────────────────────────────────────
# When SHARED_COLORSCALE=True, compute vmax in fT from all three evoked objects
# at the selected time points and pass it via topomap_args to plot_joint().
if SHARED_COLORSCALE:
    def _peak_fT(evoked, times):
        data_fT = evoked.copy().pick('mag').get_data() * 1e15
        time_idx = [np.argmin(np.abs(evoked.times - t)) for t in times]
        return np.abs(data_fT[:, time_idx]).max()

    shared_vmax = max(_peak_fT(ev, JOINT_TIMES) for ev, _ in evokeds)
    topomap_args = dict(vlim=(-shared_vmax, shared_vmax))
    print(f'Shared colorscale: ±{shared_vmax:.1f} fT')
else:
    topomap_args = {}
    print('Independent colorscale per plot (MNE default)')

# Joint plots for X, Y, Z components of the context-matched MMN.
# Comparing all three components shows how the mismatch field
# is distributed across the radial and tangential sensor axes.
JOINT_KWARGS = dict(
    times=JOINT_TIMES,
    picks='mag',
    topomap_args=topomap_args,
    show=False,
)

for ev, label in evokeds:
    fig = ev.plot_joint(title=label, **JOINT_KWARGS)
    plt.show()



## 5. Image Plots

Like `Epochs`, the `Evoked` object also has a `plot_image()` method — but with an
important difference:

- `epochs.plot_image()` — one **epoch** per row, one channel per figure
- `evoked.plot_image()` — one **channel** per row, time on the x-axis

The color at each position encodes the field amplitude at that channel and time
point. A line plot of the mean across channels is shown below the image.
This makes it easy to see which channels drive the response and how the
activity is distributed spatially over time.


In [ ]:
# evoked.plot_image() — one channel per row, time on x-axis.
# picks='mag' restricts to magnetometer channels (excludes stim, eyetracking etc.).
# We plot the context-matched MMN to highlight which channels show the mismatch
# response and when.
fig = evoked_mmn_z.plot_image(
    picks='mag',
    units=dict(mag='fT'),
    scalings=dict(mag=1e15),
    titles=dict(mag='Context-matched MMN — Z'),
    show=False,
)
plt.show()


## 6. Topographical Subplots

For sensor-level analyses, it can be useful to plot the response at each sensor
in a topographical layout.

### 6a. `plot_compare_evokeds` with `axes='topo'`

`plot_compare_evokeds()` can produce a topographical layout by passing `axes='topo'`.
This overlays conditions at each sensor position, making it easy to see where
conditions differ across the scalp. However, it can be slow with many sensors —
we restrict to Z channels (`picks=z_names`) to keep it to 64 subplots.


In [ ]:
# axes='topo' arranges one subplot per sensor in a topographical layout.
# We restrict to Z channels.
fig = mne.viz.plot_compare_evokeds(
    {'Standard (matched)': [evoked_std_matched_z],
     DEVIANT_COND:         [evoked_dev_z]},
    picks=z_names,
    axes='topo',
    show=False,
)
plt.show()


## 7. 3D Field Maps

The scalp topographies above were all projected into two-dimensional overhead
views of the field, but it is also possible to plot field maps in 3D.

This requires a **trans file** — a coordinate transform that aligns MEG sensor
positions to the head surface (derived from the MRI). The trans file is produced
during coregistration (e.g. `mne coreg`). You can compute 3D field maps *without*
a trans file, but only for estimating the field on the MEG helmet surface itself
(not the scalp or cortex).

By default, MEG sensors are used to estimate the field on the helmet surface.
Once the maps are computed with `mne.make_field_map()`, you visualise them with
`evoked.plot_field()`.

Since our OPM data has only `mag` channels (no gradiometers, no EEG), we use
the full mag evoked object — not just Z channels — so that all sensor positions
are available for the field interpolation.


In [ ]:
# make_field_map() interpolates the field across a surface from the sensor data.
# trans=    — path to the coregistration transform (.fif)
# subject=  — FreeSurfer subject name (must exist in SUBJECTS_DIR)
# origin=   — 'auto' estimates the sphere origin from the digitisation points
#
# We use evoked_dev (full mag, not Z-only) so all sensor positions contribute
# to the interpolation.
maps = mne.make_field_map(
    evoked_dev,
    trans=str(TRANS_FILE),
    subject=FS_SUBJECT,
    subjects_dir=SUBJECTS_DIR,
    origin='auto',
)

# plot_field() renders the interpolated field on the helmet or head surface.
# time= selects the time point to display (in seconds).
# Here we look at the supposed MMN period at 200ms.
evoked_dev.plot_field(maps, time=0.200)


You can also pass `meg_surf='head'` to project the MEG field onto the head
surface (rather than the helmet). This requires the head surface file
`{FS_SUBJECT}-head.fif` to exist in the bem folder.

**Prerequisite:** This file is not created automatically by `recon-all` or
`make_watershed_bem`. You need to run `mne.bem.make_scalp_surfaces()` once per
subject, which reads the watershed outer skin surface already in your bem folder
and writes the required head surface file. Run the cell below first if you
haven't already:


In [ ]:
# Run once per subject to generate the head surface file needed for meg_surf='head'.
# This reads watershed/sub-001_ses-01_outer_skin_surface from the bem folder
# and writes sub-001_ses-01-head.fif (and related files) into bem/.
# Safe to re-run with overwrite=True if needed.
mne.bem.make_scalp_surfaces(
    subject=FS_SUBJECT,
    subjects_dir=SUBJECTS_DIR,
    overwrite=True,
    verbose=True,
)

# RuntimeWarnings about topological defects (e.g. "32 / 14998 vertices have fewer
# than three neighboring triangles") can be safely ignored here. A small number of
# defective vertices is common in watershed-derived surfaces and does not affect the
# quality of the field map. MNE will raise these as errors later if they are
# genuinely problematic — for now, the "[done]" message confirms the surfaces were
# created successfully.

In [ ]:
maps_head = mne.make_field_map(
    evoked_dev,
    trans=str(TRANS_FILE),
    subject=FS_SUBJECT,
    subjects_dir=SUBJECTS_DIR,
    meg_surf='head',       # project onto head surface rather than helmet
    head_source='head-dense',   # dense tessellation — passes topology checks cleanly
    origin='auto',
)
evoked_dev.plot_field(maps_head, time=0.200)


## 8. Amplitude and Latency Measures

Visualisation tells you *where* and *when* responses occur — but for reporting
results you also need numbers: the latency of a peak, the amplitude at a given
channel, or the mean amplitude across a time window.

MNE provides two main approaches:

- `evoked.get_peak()` — returns the channel and time point of the maximum response
  within a specified window, along with the amplitude at that point
- Manual indexing of `evoked.data` — for mean amplitude over a time window
  across a set of channels

We focus on the **context-matched MMN** difference wave, with a search window of
175–225 ms based on visual inspection of the GFP.

### 8a. Peak latency and amplitude with `get_peak()`

`get_peak()` searches a specified time window and returns the channel with the
largest response and its latency. Key parameters:

- `ch_type` — must be specified when the evoked contains only one channel type
- `tmin` / `tmax` — search window in seconds
- `mode` — `'abs'` (default): largest absolute value; `'pos'`: largest positive;
  `'neg'`: largest negative
- `return_amplitude=True` — also return the amplitude value at the peak

**Important:** `get_peak()` finds the global maximum in the window — it does not
guarantee that the returned time point is a true local peak. Always visually
inspect the data to confirm the window actually contains the component of interest.


In [ ]:
# get_peak() returns (channel_name, latency_s, amplitude) when return_amplitude=True.
# mode='abs' finds the channel with the largest absolute value — appropriate for
# OPM-MEG where the dipolar field produces both positive and negative peaks.
ch_mmn, lat_mmn, amp_mmn = evoked_mmn_z.get_peak(
    ch_type='mag',
    tmin=0.175,
    tmax=0.225,
    mode='abs',
    return_amplitude=True,
)
print('MMN peak (context-matched):')
print(f'  Channel   : {ch_mmn}')
print(f'  Latency   : {lat_mmn * 1000:.1f} ms')
print(f'  Amplitude : {amp_mmn * 1e15:.2f} fT')


### 8b. Mean amplitude over a time window

Peak amplitude is sensitive to noise — a single noisy time point can dominate.
Mean amplitude over a pre-defined time window is more robust and is the standard
measure in most ERP/ERF studies.

The approach is to slice `evoked.data` directly using `time_as_index()` to convert
the time window boundaries to sample indices, then take the mean over the time
dimension. We can do this for a single channel, across all channels, or summarise
with GFP (RMS) to avoid signal cancellation across the sensor array.


In [ ]:
# Mean amplitude in the MMN window — more robust than peak amplitude.
# time_as_index() converts seconds to sample indices.
t_idx = evoked_mmn_z.time_as_index([0.175, 0.225])

# ── Single peak channel ───────────────────────────────────────────────────────
# Use the MMN peak channel found above.
ch_idx   = evoked_mmn_z.ch_names.index(ch_mmn)
mean_amp = evoked_mmn_z.data[ch_idx, t_idx[0]:t_idx[1]].mean() * 1e15
print(f'Mean amplitude at {ch_mmn} (175–225 ms): {mean_amp:.2f} fT')

# ── Mean across all Z channels ────────────────────────────────────────────────
# Average first across time, then across channels.
window_data  = evoked_mmn_z.data[:, t_idx[0]:t_idx[1]]   # shape: (n_channels, n_times)
mean_per_ch  = window_data.mean(axis=1)                   # mean over time → (n_channels,)
mean_all_chs = mean_per_ch.mean() * 1e15                  # mean over channels
print(f'Mean amplitude across all Z channels (175–225 ms): {mean_all_chs:.2f} fT')

# ── GFP (RMS) in the window ───────────────────────────────────────────────────
# GFP summarises response strength across all channels without signal cancellation.
gfp_mmn      = np.sqrt(np.mean(evoked_mmn_z.data ** 2, axis=0)) * 1e15
gfp_mean_mmn = gfp_mmn[t_idx[0]:t_idx[1]].mean()
print(f'Mean GFP of context-matched MMN (175–225 ms): {gfp_mean_mmn:.2f} fT')


### 8c. Visualising the peak on the waveform

It is good practice to plot the detected peak on the waveform to confirm it
falls in the right place — `get_peak()` finds the global maximum in the window
and will not warn you if that maximum is on the rising or falling edge of a
noise artefact rather than a true component peak.


In [ ]:
# Plot the MMN peak channel waveform and mark the detected peak.
ch_idx   = evoked_mmn_z.ch_names.index(ch_mmn)
times_ms = evoked_mmn_z.times * 1000
data_fT  = evoked_mmn_z.data[ch_idx] * 1e15

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(times_ms, data_fT, color='tab:blue', lw=1.5, label=f'{ch_mmn}')
ax.axvspan(-0.200, 0,   color='gray',   alpha=0.15, zorder=0)
ax.axvspan(175,  225, color='yellow', alpha=0.25, zorder=0, label='search window (175–225 ms)')
ax.axvline(0,    color='black', lw=0.8, ls='--')
ax.axvline(lat_mmn * 1000, color='red', lw=1.5, ls='--',
           label=f'peak: {lat_mmn*1000:.1f} ms, {amp_mmn*1e15:.1f} fT')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Amplitude (fT)')
ax.set_title(f'Context-matched MMN — {ch_mmn}: peak detection')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 8d. Top 5 channels by MMN amplitude

This builds on the RMS channel-ranking exercise from notebook 2.1 Section 5b,
where we identified the top 5 M100 channels using the same approach. Here we
apply the same ranking to the MMN window and extend it with per-channel peak
latency reporting and waveform visualisation.

Rather than reporting a single peak channel, it can be useful to identify the
top-responding channels and report their latencies and amplitudes individually.
We rank channels by RMS in the MMN window, then call `get_peak()` once per channel.
The waveform plot lets you confirm that all five channels show a coherent MMN shape.
And you can plot those sensor locations on the sensor map to see if they fall where expected.

In [ ]:
# ── Rank channels by RMS in the MMN window ───────────────────────────────────
t_idx    = evoked_mmn_z.time_as_index([0.175, 0.225])
window   = evoked_mmn_z.data[:, t_idx[0]:t_idx[1]]
rms      = np.sqrt(np.mean(window ** 2, axis=1))
top5_idx   = np.argsort(rms)[::-1][:5]
top5_names = [evoked_mmn_z.ch_names[i] for i in top5_idx]

# ── Peak latency and amplitude for each top channel ──────────────────────────
print('Top 5 MMN channels (175–225 ms):')
print(f'  {"Channel":<20s}  {"Latency":>10s}  {"Amplitude":>12s}')
print(f'  {"-"*20}  {"-"*10}  {"-"*12}')
results = []
for ch in top5_names:
    ch_evoked = evoked_mmn_z.copy().pick_channels([ch])
    _, lat, amp = ch_evoked.get_peak(
        ch_type='mag',
        tmin=0.175, tmax=0.225,
        mode='abs',
        return_amplitude=True,
    )
    results.append((ch, lat, amp))
    print(f'  {ch:<20s}  {lat*1000:>8.1f} ms  {amp*1e15:>10.2f} fT')

# ── Waveform plot for all 5 channels ─────────────────────────────────────────
times_ms = evoked_mmn_z.times * 1000
fig, ax  = plt.subplots(figsize=(9, 3.5))

colors = plt.cm.tab10.colors
for (ch, lat, amp), color in zip(results, colors):
    ch_idx = evoked_mmn_z.ch_names.index(ch)
    ax.plot(times_ms, evoked_mmn_z.data[ch_idx] * 1e15,
            color=color, lw=1.5, label=ch)
    ax.axvline(lat * 1000, color=color, lw=1, ls=':', alpha=0.7)

ax.axvspan(-0.200, 0,   color='gray',   alpha=0.15, zorder=0)
ax.axvspan(175,    225, color='yellow', alpha=0.20, zorder=0, label='search window')
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Amplitude (fT)')
ax.set_title('Context-matched MMN — top 5 Z channels')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

# Plot the locations of the top 5 MMN channels on a sensor map.
fig = evoked_mmn_z.plot_sensors(
    show_names=top5_names,
    title='Top 5 MMN channels (175–225 ms)',
    show=False,
)
plt.show()

# Topomap averaged over four windows spanning the epoch.
# average= sets the window half-width in seconds (0.050 = ±25 ms around each center).
fig = evoked_mmn_z.plot_topomap(
    times=[0.000, 0.100, 0.200, 0.300],
    ch_type='mag',
    average=0.050,
    time_unit='ms',
    colorbar=True,
    show=False,
)
fig.suptitle('Context-matched MMN — Z channels: field averaged over 50 ms windows', y=1.02)
plt.show()

## 9. Summary

| Task | Method |
|------|--------|
| Butterfly plot | `evoked.plot(gfp=True)` |
| Highlight time windows | `evoked.plot(highlight=[(t0, t1), ...])` |
| Overlay conditions | `mne.viz.plot_compare_evokeds({label: [evo], ...})` |
| Topomap at time points | `evoked.plot_topomap(times=[...])` |
| Topomap averaged over window | `evoked.plot_topomap(times=[...], average=0.05)` |
| Interactive topo from butterfly | Click on butterfly in `%matplotlib qt` mode |
| Joint plot (butterfly + topos) | `evoked.plot_joint(times=[...])` |
| Image plot (channels × time) | `evoked.plot_image(picks='mag')` |
| Topographical subplots | `mne.viz.plot_compare_evokeds({...}, axes='topo')` |
| 3D field map on helmet | `mne.make_field_map(evoked, trans=...) → evoked.plot_field(maps)` |
| 3D field map on head surface | `mne.make_field_map(..., meg_surf='head', head_source='head-dense')` |
| Generate head surface file | `mne.bem.make_scalp_surfaces(subject=..., subjects_dir=...)` |
| Peak channel and latency | `evoked.get_peak(ch_type='mag', tmin=..., tmax=..., return_amplitude=True)` |
| Mean amplitude in window | `evoked.data[ch_idx, t0:t1].mean()` via `time_as_index()` |
| GFP in window | `np.sqrt(np.mean(evoked.data**2, axis=0))[t0:t1].mean()` |

### Key OPM-MEG reminders

- Always use `exclude='bads'` when picking channels for analysis
- Z (radial) channels give the most interpretable topographies for auditory responses;
  X and Y (tangential) carry complementary spatial information and are shown in the joint plots
- The dipolar field pattern in Z topographies reflects the physics of magnetic dipoles,
  not EEG-style voltage polarity
- `gfp=True` in `evoked.plot()` adds a thick green RMS trace — equivalent to Global
  Field Power and a useful single-number summary of response strength at each time point
- `plot_compare_evokeds` confidence bands and individual channel traces only appear
  when multiple evoked objects are passed per condition (e.g. one per subject in a group study)
- The MMN difference wave (`evoked_mmn`) uses the **context-matched** standard — the
  standard trial immediately preceding each deviant — for equal trial counts and
  better adaptation control than the conventional contrast
- `make_scalp_surfaces` must be run once per subject before `meg_surf='head'` field maps;
  use `head_source='head-dense'` if medium/sparse tessellations have topological defects
- `get_peak()` finds the global maximum in a window — always plot the waveform with the
  detected peak marked to confirm it falls on a true component and not noise
- Mean amplitude over a time window is more robust than peak amplitude for noisy data;
  GFP (RMS across channels) avoids signal cancellation across the sensor array

### Next steps
- **Time-frequency analysis** — Morlet wavelets, spectrograms, induced vs evoked power
